# ACCT 3343 — Chapter 2: GPA and B/M Rankings

Today’s workflow: **company data → ask AI for code → paste → run → inspect**.


## 1. Load the data

Press **Run**. The classroom dataset loads automatically.


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv"
df = pd.read_csv(url)

print("Data loaded!")
df.head()


## 2. Keep each company’s latest fiscal period

`df` is our table of company data.


In [ ]:
df["fiscal_period_end"] = pd.to_datetime(df["fiscal_period_end"], errors="coerce")

numeric_columns = [
    "gross_profit",
    "total_assets",
    "stockholders_equity",
    "ordinary_shares_number",
    "fiscal_period_end_close",
]
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

# Keep the latest available fiscal period for each company.
df = (
    df.sort_values(["ticker", "fiscal_period_end"], ascending=[True, False])
      .drop_duplicates("ticker", keep="first")
      .copy()
)

# Market value of equity at the fiscal date.
df["market_equity"] = (
    df["ordinary_shares_number"] * df["fiscal_period_end_close"]
)

columns_to_keep = [
    "ticker", "company_name", "sector", "industry",
    "fiscal_period_end", "fiscal_year",
    "gross_profit", "total_assets", "stockholders_equity",
    "ordinary_shares_number", "fiscal_period_end_close", "market_equity",
]
df = df[columns_to_keep]

assert df["ticker"].nunique() == len(df)
assert len(df) == 50

print("Companies:", df["ticker"].nunique())
df.head()


Market equity is fiscal-period shares × fiscal-period closing price. Yahoo’s annual share count may not be perfectly date-matched; missing values stay missing.


## 3. Gross profitability (GPA)

Higher GPA is better.

Ask Kai:

> **df: create GPA = gross_profit / total_assets. Code only.**


In [ ]:
# Paste Kai's code here, then press Run


### Reference solution — use only if needed


In [ ]:
df["GPA"] = df["gross_profit"] / df["total_assets"]


In [ ]:
df[["ticker", "GPA"]].sort_values("GPA", ascending=False)


## 4. Book-to-market (B/M)

Higher B/M is better.

Ask Kai:

> **df: create BM = stockholders_equity / market_equity. Code only.**


In [ ]:
# Paste Kai's code here, then press Run


### Reference solution — use only if needed


In [ ]:
df["BM"] = df["stockholders_equity"] / df["market_equity"]


In [ ]:
df[["ticker", "BM"]].sort_values("BM", ascending=False)


## 5. Rank GPA and B/M

Ask Kai:

> **Rank GPA and BM. High = good, best = 1. Create GPA_Rank, BM_Rank. Code only.**


In [ ]:
# Paste Kai's code here, then press Run


### Reference solution — use only if needed


In [ ]:
df["GPA_Rank"] = df["GPA"].rank(ascending=False, method="min")
df["BM_Rank"] = df["BM"].rank(ascending=False, method="min")


In [ ]:
df[["ticker", "GPA", "GPA_Rank", "BM", "BM_Rank"]]


## 6. Combine the ranks

Lower combined rank is better.

Ask Kai:

> **Combined_Rank = GPA_Rank + BM_Rank. Sort low→high. Show ticker, GPA, BM, ranks. Code only.**


In [ ]:
# Paste Kai's code here, then press Run


### Reference solution — use only if needed


In [ ]:
df["Combined_Rank"] = df["GPA_Rank"] + df["BM_Rank"]

student_result = (
    df[["ticker", "GPA", "BM", "GPA_Rank", "BM_Rank", "Combined_Rank"]]
    .sort_values("Combined_Rank", ascending=True, na_position="last")
)
student_result


## 7. Final table


In [ ]:
final_table = (
    df[[
        "ticker", "company_name", "sector", "industry",
        "GPA", "GPA_Rank", "BM", "BM_Rank", "Combined_Rank",
    ]]
    .sort_values("Combined_Rank", ascending=True, na_position="last")
)

final_table


## Class discussion

Look at the top 10 firms.

- What types of companies rank highly?
- Is a high-GPA company necessarily a high-B/M company?
- Why might the two measures identify different firms?
- What happens when we combine the rankings?
